In [1]:
import pandas as pd
import numpy as np

In [3]:
clean_df = pd.read_csv(r"C:\Users\nivet\retail-orders-cleaned.csv")
print("File Uploaded Successfully")

print("Rows : ", len(clean_df))
print("Columns : ", len(clean_df.columns))

File Uploaded Successfully
Rows :  7
Columns :  9


In [4]:
clean_df.head()

,order_id,order_date,customer_segment,city,category,quantity,unit_price,discount_pct,payment_status
0,RT-1001,2026-01-03,Student,Chennai,Learning Kit,2,799,10.0,Paid
1,RT-1002,2026-01-03,Fresher,Bengaluru,Course Access,1,1499,0.0,Paid
2,RT-1003,2026-01-05,Student,Chennai,Course Access,1,1499,0.0,Pending
3,RT-1004,2026-01-07,Professional,Hyderabad,Learning Kit,3,799,5.0,Paid
4,RT-1008,2026-01-14,Fresher,Bengaluru,Course Access,2,1499,0.0,Pending


In [5]:
#COMPLETENESS CHECK

req_columns =[
    "order_id", "order_date", "city", "category", "quantity", "unit_price", "payment_status"
]

missing_profile = pd.DataFrame({
    "column" : req_columns,
    "missing_count" : clean_df[req_columns].isna().sum(),
    "missing_percentage" : (clean_df[req_columns].isna().mean() * 100).round(2)
})
missing_profile

,column,missing_count,missing_percentage
order_id,order_id,0,0.0
order_date,order_date,0,0.0
city,city,0,0.0
category,category,0,0.0
quantity,quantity,0,0.0
unit_price,unit_price,0,0.0
payment_status,payment_status,0,0.0


In [6]:
duplicate_count = clean_df["order_id"].duplicated().sum()

print("Duplicate order IDs:", duplicate_count)

Duplicate order IDs: 0


In [7]:
print("Unique order IDs:", clean_df["order_id"].nunique())
print("Total records:", len(clean_df))

Unique order IDs: 7
Total records: 7


In [8]:
#QUANTITY VALIDITY 

invalid_quantity = clean_df[
    clean_df["quantity"].isna() |
    (clean_df["quantity"] <= 0) |
    (clean_df["quantity"] % 1 != 0)
]

print("Invalid quantity records:", len(invalid_quantity))

Invalid quantity records: 0


In [9]:
#UNIT PRICE VALIDITY

invalid_price = clean_df[
    clean_df["unit_price"].isna() |
    (clean_df["unit_price"] < 0)
]

print("Invalid unit price records:", len(invalid_price))

Invalid unit price records: 0


In [10]:
#DISCOUNT PRICE VALIDITY

invalid_discount = clean_df[
    clean_df["discount_pct"].isna() |
    (clean_df["discount_pct"] < 0) |
    (clean_df["discount_pct"] > 100)
]

print("Invalid discount records:", len(invalid_discount))

Invalid discount records: 0


In [11]:
#CATEGORY VALIDITY CHECK

allowed_categories = [
    "Learning Kit",
    "Course Access",
    "Mentor Session"
]

invalid_categories = clean_df[
    ~clean_df["category"].isin(allowed_categories)
]

print("Invalid category records:", len(invalid_categories))

Invalid category records: 0


In [12]:
#CUSTOMER SEGMENT VALIDITY 
allowed_segments = [
    "Student",
    "Fresher",
    "Professional"
]

invalid_segments = clean_df[
    ~clean_df["customer_segment"].isin(allowed_segments)
]

print("Invalid customer segments:", len(invalid_segments))

Invalid customer segments: 0


In [13]:
#PAYMENT STATUS VALIDITY

allowed_payment_status = [
    "Paid",
    "Pending",
    "Failed",
    "Refunded"
]

invalid_payment = clean_df[
    ~clean_df["payment_status"].isin(
        allowed_payment_status
    )
]

print("Invalid payment statuses:", len(invalid_payment))

Invalid payment statuses: 0


In [14]:
#DATA VALIDITY CHECK

clean_df["order_date"] = pd.to_datetime(
    clean_df["order_date"],
    errors="coerce"
)

today = pd.Timestamp.today().normalize()

invalid_dates = clean_df[
    clean_df["order_date"].isna() |
    (clean_df["order_date"] < pd.Timestamp("2025-01-01")) |
    (clean_df["order_date"] > today)
]

print("Invalid date records:", len(invalid_dates))

Invalid date records: 0


In [15]:
#FRESHNESS CHECK 
latest_order_date = clean_df["order_date"].max()

days_since_latest = (
    today - latest_order_date
).days

print("Latest order date:", latest_order_date.date())
print("Days since latest order:", days_since_latest)

Latest order date: 2026-01-18
Days since latest order: 223


In [17]:
#=========================================================== 
#QUALITY REPORT
#===========================================================

quality_report = pd.DataFrame({
    "Dimension": [
        "Completeness",
        "Uniqueness",
        "Quantity Validity",
        "Unit Price Validity",
        "Discount Validity",
        "Category Validity",
        "Customer Segment Validity",
        "Payment Status Validity",
        "Date Validity"
    ],
    "Issue Count": [
        clean_df[req_columns].isna().sum().sum(),
        duplicate_count,
        len(invalid_quantity),
        len(invalid_price),
        len(invalid_discount),
        len(invalid_categories),
        len(invalid_segments),
        len(invalid_payment),
        len(invalid_dates)
    ]
})

quality_report

,Dimension,Issue Count
0,Completeness,0
1,Uniqueness,0
2,Quantity Validity,0
3,Unit Price Validity,0
4,Discount Validity,0
5,Category Validity,0
6,Customer Segment Validity,0
7,Payment Status Validity,0
8,Date Validity,0


In [18]:
# QUALITY STATUS 

quality_report["Status"] = np.where(
    quality_report["Issue Count"] == 0,
    "PASS",
    "FAIL"
)

quality_report

,Dimension,Issue Count,Status
0,Completeness,0,PASS
1,Uniqueness,0,PASS
2,Quantity Validity,0,PASS
3,Unit Price Validity,0,PASS
4,Discount Validity,0,PASS
5,Category Validity,0,PASS
6,Customer Segment Validity,0,PASS
7,Payment Status Validity,0,PASS
8,Date Validity,0,PASS


In [20]:
#SAVE THE REPORT

quality_report.to_csv(
    "data_quality_report.csv", index = False )